# Notebook 1: Sequence Collection

**Pipeline step:** Fetch hydrazine synthetase sequences from NCBI and build a structured database.

**Overview:** In this notebook you will learn the Python fundamentals needed for bioinformatics scripting, understand how NCBI stores protein data, and explore the 24-entry HS database that was assembled in AAR-COMP-012. By the end you will be ready to extend that database with new entries of your own.

---

## Learning objectives

- Understand what Python is and why biologists use it
- Recognise and use variables, lists, dictionaries, and functions in Python
- Create and read CSV files with the pandas library
- Understand what NCBI, accession codes, and GenBank records are
- Fetch a protein record programmatically with Biopython
- Explore the existing HS database and design your own

---

## 1. What is Python and why do biologists use it?

Python is a programming language — a precise way of giving instructions to a computer. Unlike a graphical program like ChemDraw, Python lets you automate repetitive tasks: instead of copy-pasting 24 protein sequences one by one, you write a short script that does it in seconds.

For bioinformatics, Python is the dominant tool because:
- It reads like near-English, making it approachable for non-computer-scientists.
- Powerful free libraries exist for biology (Biopython), data handling (pandas), and plotting (matplotlib).
- It runs on any operating system, including HPC clusters.

---

## 2. Python fundamentals

### 2.1 Variables

A variable is a named container for a value. Think of it as a labelled Eppendorf tube.

```python
enzyme_name = "PyrN"          # a string (text)
sequence_length = 668          # an integer (whole number)
confidence = 0.974             # a float (decimal)
is_difunctional = True         # a boolean (True/False)
```

### 2.2 Lists

A list holds multiple values in order — like a rack of numbered tubes.

```python
enzymes = ["PyrN", "Spb40", "Tri28", "ForJ"]
print(enzymes[0])   # prints "PyrN" (Python counts from 0)
print(len(enzymes)) # prints 4
```

### 2.3 Dictionaries

A dictionary maps keys to values — like a CAS number lookup table where each CAS number gives you back the compound name and molecular weight.

```python
enzyme_info = {
    "name":     "PyrN",
    "organism": "Streptomyces candidus",
    "length":   668,
}
print(enzyme_info["organism"])   # prints 'Streptomyces candidus'
```

### 2.4 Functions

A function is a named, reusable recipe. You define it once, call it many times.

```python
def molecular_weight(sequence):
    """Estimate MW of a protein from its amino acid count."""
    avg_aa_mw = 110  # average daltons per amino acid
    return len(sequence) * avg_aa_mw

mw = molecular_weight("MTVPR")  # calls the function
print(mw)   # prints 550
```

In [1]:
# ── Run this cell to practice Python basics ─────────────────────────────

# Define a variable
enzyme_name = "PyrN"
sequence_length = 668

# Print them — .format() inserts values into a string
print("Enzyme: {}  |  Length: {} aa".format(enzyme_name, sequence_length))

# A list of the di-domain enzymes in our database
di_domain = ["Spb40", "Tri28", "Aza12", "NngM", "DobE",
             "PyrN", "ThzN", "Apy9", "Por11", "ForJ"]

print("Number of di-domain enzymes: {}".format(len(di_domain)))
print("First entry: {}".format(di_domain[0]))

# A dictionary for one enzyme
pyrn = {"name": "PyrN", "organism": "Streptomyces candidus", "length": 668}
print("Organism: {}".format(pyrn["organism"]))

Enzyme: PyrN  |  Length: 668 aa
Number of di-domain enzymes: 10
First entry: Spb40
Organism: Streptomyces candidus


---

## 3. CSV files and pandas

A **CSV (comma-separated values)** file is a plain-text table where columns are separated by commas. It can be opened in Excel, but crucially it can also be read and written by Python scripts — making it ideal for pipelines. Excel's `.xlsx` format requires special software to parse; CSV is universal.

**pandas** is the Python library for working with tabular data. Its central object is a `DataFrame` — a table with labelled rows and columns.

> **Key concept:** A `DataFrame` is like a spreadsheet inside Python. Each column has a name; each row has an index. You can filter, sort, and compute on it in one line.

In [2]:
# ── Import pandas ──────────────────────────────────────────────────────
import pandas as pd   # 'pd' is the conventional short name

# Create a tiny DataFrame from scratch
# Each key becomes a column; each list becomes that column's values
small_table = pd.DataFrame({
    "Name":     ["PyrN", "Spb40", "ForJ"],
    "Organism": ["S. candidus", "Streptomyces sp.", "S. kaniharaensis"],
    "Length":   [668, 661, 674],
})

print(small_table)          # print the table
print()
print(small_table.dtypes)   # check column data types

    Name          Organism  Length
0   PyrN       S. candidus     668
1  Spb40  Streptomyces sp.     661
2   ForJ  S. kaniharaensis     674

Name          str
Organism      str
Length      int64
dtype: object


In [ ]:
# ── Save the DataFrame to a CSV file ──────────────────────────────────
small_table.to_csv("/tmp/my_test.csv", index=False)  # index=False avoids a row-number column

# ── Read it back ──────────────────────────────────────────────────────
loaded = pd.read_csv("/tmp/my_test.csv")
print(loaded)

---

## 4. The HS database schema

The HS database CSV (`HS_database.csv`) has 12 columns, matching the fields in a GenBank protein record:

| Column | What it contains |
|---|---|
| Hydrazine synthase name | Short name we assign (e.g. PyrN) |
| GenBank ID | Unique versioned identifier (e.g. QDO67124.1) |
| Accession Code | Same or base accession |
| UniProt ID | Cross-reference to UniProtKB (if available) |
| Organism | Source organism |
| Year Deposited | Date the record was submitted |
| Reference(s) | Literature citation(s) |
| Domain Architecture | Di-domain / Cupin only / MetRS only / Unknown |
| Sequence Length | Number of amino acids |
| Cupin domain range | e.g. 28-101 (1-based, inclusive) |
| MetRS domain range | e.g. 121-646 |
| Sequence | Full amino acid sequence |

In [ ]:
# ── Load the existing HS database ─────────────────────────────────────
DB_PATH = "./HS_database.csv"

db = pd.read_csv(DB_PATH)   # read the CSV into a DataFrame

# Show dimensions and the first two rows
print("Shape: {} rows x {} columns".format(db.shape[0], db.shape[1]))
print()
db.head(2)   # .head(n) shows the first n rows

Shape: 24 rows x 12 columns



,Hydrazine synthase name,GenBank ID,Accession Code,UniProt ID,Organism,Year Deposited,Reference(s),Domain Architecture,Sequence Length,Cupin domain range,MetRS domain range,Sequence
0,Spb40,BAW27704.1,BAW27704.1,NaN,Streptomyces sp. SoC090715LN-17,12-MAY-2017,"Matsuda,K., Hasebe,F., Shiwa,Y., Kanesaki,Y., ...",Di-domain,661,28-101,121-646,MILNQYSESELSSAFGIEMSPVEGLGVGAGWGRVAPGASSHPDRHD...
1,Kit2,WP_184911086.1,WP_184911086.1,NaN,Kitasatospora gansuensis,15-MAR-2026,NaN,MetRS only,512,NaN,3-442,MSKRVAVISPAPTANGDLHLGHLAGPFLAADVYTRYQRATGRETVF...


In [4]:
# ── Explore the database ───────────────────────────────────────────────

# How many of each domain architecture?
print("Domain architecture counts:")
print(db["Domain Architecture"].value_counts())
print()

# Sequence length statistics
print("Sequence length statistics:")
print(db["Sequence Length"].describe())
print()

# List all organism names
print("Organisms:")
for org in db["Organism"].tolist():
    print("  " + org)

Domain architecture counts:
Domain Architecture
Di-domain     10
MetRS only     7
Cupin only     7
Name: count, dtype: int64

Sequence length statistics:
count     24.000000
mean     472.666667
std      243.873571
min      114.000000
25%      139.000000
50%      531.500000
75%      668.750000
max      760.000000
Name: Sequence Length, dtype: float64

Organisms:
  Streptomyces sp. SoC090715LN-17
  Kitasatospora gansuensis
  Streptomyces tumemacerans
  Kitasatospora aureofaciens
  Glycomyces harbinensis
  Streptomyces yunnanensis
  Nocardia sp. CS682
  Streptomyces candidus
  Streptomyces tumemacerans
  Streptomyces coerulescens
  Streptomyces sp. MSD090630SC-05
  Pseudomonas orientalis
  Streptomyces kaniharaensis
  Streptomyces
  Streptomyces
  Bacillus
  Bacillus
  Colwellia sp.
  Colwellia sp.
  Kitasatospora gansuensis
  Ralstonia solanacearum species complex
  Ralstonia solanacearum
  Corallococcus llansteffanensis
  Corallococcus llansteffanensis


In [5]:
# ── Filter: show only di-domain enzymes ────────────────────────────────
# Boolean indexing: db[condition] returns rows where condition is True

di_domain_df = db[db["Domain Architecture"] == "Di-domain"]

# Select only the informative columns to display
cols_to_show = ["Hydrazine synthase name", "Organism",
                "Sequence Length", "Cupin domain range", "MetRS domain range"]
print(di_domain_df[cols_to_show].to_string(index=False))

Hydrazine synthase name                        Organism  Sequence Length Cupin domain range MetRS domain range
                  Spb40 Streptomyces sp. SoC090715LN-17              661             28-101            121-646
                  Tri28      Kitasatospora aureofaciens              678             28-101            121-663
                  Aza12          Glycomyces harbinensis              745             28-101            121-573
                   NngM        Streptomyces yunnanensis              744             29-104            124-646
                   DobE              Nocardia sp. CS682              671             26-101            122-648
                   PyrN           Streptomyces candidus              668             40-108            131-640
                   ThzN       Streptomyces coerulescens              760             36-106            126-653
                   Apy9 Streptomyces sp. MSD090630SC-05              650             32-106            127-637
 

---

## 5. What is NCBI and what does a GenBank record contain?

**NCBI (National Center for Biotechnology Information)** is the primary public repository for biological sequence data. Every protein sequence deposited in GenBank receives a unique **accession code** (e.g. `QDO67124.1`). The number after the dot is the version — if the submitter corrects an error, the version increments.

A **GenBank protein record** contains:
- Metadata: organism, taxonomy, submitter, date
- Amino acid sequence
- **Feature annotations**: CDS, Region (domain calls from NCBI CDD), Site features
- Cross-references: UniProtKB, PDB, etc.
- Literature references

> **Key concept:** The `Region` features in a GenBank record are how the script detects cupin and MetRS domain boundaries automatically — without any manual annotation.

---

## 6. Biopython and `Entrez.efetch`

**Biopython** is the standard Python library for bioinformatics. Its `Bio.Entrez` module provides functions that communicate directly with NCBI databases over the internet.

The key function is `Entrez.efetch` — it downloads a record in a specified format (e.g. GenBank flat-file) and returns a file-like object that `SeqIO.read` can parse.

Here is the core of `fetch_hs_entry.py`, annotated line by line:

In [ ]:
# ── Biopython imports ─────────────────────────────────────────────────
from Bio import Entrez, SeqIO   # Entrez = NCBI interface; SeqIO = sequence I/O

# NCBI requires an email address so they can contact you if the server is
# overloaded. Replace with your own email.
Entrez.email = "your.email@example.com"


def fetch_record(accession):
    """Fetch and parse a GenBank protein record using Biopython."""
    # efetch opens a connection to NCBI and downloads the record
    # db="protein"  : search the protein database
    # rettype="gb"  : return GenBank flat-file format
    # retmode="text": plain text (not XML)
    handle = Entrez.efetch(db="protein", id=accession, rettype="gb", retmode="text")

    # SeqIO.read parses the file-like handle into a SeqRecord object
    record = SeqIO.read(handle, "genbank")
    handle.close()   # always close the handle to free the connection
    return record


print("fetch_record() function defined — run the next cell to use it.")

In [ ]:
# ── Fetch PyrN (QDO67124.1) and inspect the record ────────────────────
# This requires an internet connection on the HPC login node

record = fetch_record("QDO67124.1")

# Basic metadata stored in record.annotations
print("ID:       {}".format(record.id))
print("Name:     {}".format(record.name))
print("Organism: {}".format(record.annotations.get("organism", "NaN")))
print("Date:     {}".format(record.annotations.get("date", "NaN")))
print("Length:   {} aa".format(len(record.seq)))
print()

# The sequence itself
print("First 60 aa: {}".format(str(record.seq)[:60]))

---

## 6b. Importing from your own Excel spreadsheet

If you have already collected sequences in an Excel file, you do not need to fetch them one by one from NCBI. pandas can read `.xlsx` files directly using `pd.read_excel()` — it requires the `openpyxl` library, which is already installed in the conda environment.

The workflow is:
1. Read your Excel file into a DataFrame
2. Print the column names to see what you have
3. Rename your columns to match the 12-column HS database schema
4. Save as a CSV file

> **Key concept:** `df.rename(columns={...})` lets you map old column names to new ones without touching the data.

In [ ]:
# ── Step 1: read your Excel file ──────────────────────────────────────
import pandas as pd

# Change this to the path of your Excel file
EXCEL_PATH = "/scratch/p318738/SSA/AAR-COMP-013/my_HS_database.xlsx"

excel_df = pd.read_excel(EXCEL_PATH)

# ── Step 2: inspect what columns you have ─────────────────────────────
print("Your Excel columns:")
for col in excel_df.columns:
    print("  '{}'".format(col))
print()
print("Rows: {}".format(len(excel_df)))
excel_df.head(3)

In [ ]:
# ── Step 3: rename your columns to match the HS database schema ────────
# Edit the dictionary below: left side = your Excel column name,
#                             right side = the required HS database column name
COLUMN_MAP = {
    "Name":              "Hydrazine synthase name",
    "GenBank":           "GenBank ID",
    "Accession":         "Accession Code",
    "UniProt":           "UniProt ID",
    "Organism":          "Organism",
    "Year":              "Year Deposited",
    "Reference":         "Reference(s)",
    "Domain":            "Domain Architecture",
    "Length":            "Sequence Length",
    "Cupin range":       "Cupin domain range",
    "MetRS range":       "MetRS domain range",
    "Sequence":          "Sequence",
}

# Only rename columns that actually exist in your file
rename_map = {k: v for k, v in COLUMN_MAP.items() if k in excel_df.columns}
my_db = excel_df.rename(columns=rename_map)

# ── Step 4: keep only the 12 required columns (add NaN for missing ones)
FIELDS = [
    "Hydrazine synthase name", "GenBank ID", "Accession Code", "UniProt ID",
    "Organism", "Year Deposited", "Reference(s)", "Domain Architecture",
    "Sequence Length", "Cupin domain range", "MetRS domain range", "Sequence",
]
for col in FIELDS:
    if col not in my_db.columns:
        my_db[col] = None
my_db = my_db[FIELDS]

# ── Step 5: save as CSV ───────────────────────────────────────────────
MY_DB_PATH = "/scratch/p318738/SSA/AAR-COMP-013/my_HS_database.csv"
my_db.to_csv(MY_DB_PATH, index=False)

print("Saved {} entries to: {}".format(len(my_db), MY_DB_PATH))
my_db.head()

In [ ]:
# ── Inspect the domain Region features ───────────────────────────────
# Features are annotations in the GenBank record (CDS, Region, Site, etc.)

for feature in record.features:
    if feature.type == "Region":   # only look at Region annotations
        # feature.location.start is 0-based; add 1 for 1-based inclusive
        start = int(feature.location.start) + 1
        end   = int(feature.location.end)
        # Qualifiers are a dictionary of annotation key-value pairs
        name  = feature.qualifiers.get("region_name", ["?"])[0]
        note  = feature.qualifiers.get("note", ["?"])[0][:80]
        print("{:<30} {}-{}   note: {}".format(name, start, end, note))

---

## 7. Running `fetch_hs_entry.py` from the terminal

The full script `fetch_hs_entry.py` automates everything above — fetch, parse, extract domain ranges, and append to the CSV — in a single command.

**Activate the environment first** (run once per session):

```bash
source /cvmfs/hpc.rug.nl/versions/2023.01/rocky8/x86_64/amd/zen3/software/Miniconda3/22.11.1-1/etc/profile.d/conda.sh
conda activate comp_analysis
```

**Then run the script:**

```bash
cd /scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step1/

# Syntax: python fetch_hs_entry.py <ACCESSION> [NAME]
python fetch_hs_entry.py BAW27704.1 Spb40
```

The script prints the fetched entry to the terminal and appends it to `HS_database.csv`. If the accession is already in the file, it skips silently.

> **Key concept:** Command-line scripts are the backbone of bioinformatics pipelines. A notebook is great for exploration; a script is great for automation. This tutorial uses both.

---

## 8. Building your own HS database

For your own project you will create a separate CSV with the same 12-column schema. This keeps your data cleanly separated from the AAR-COMP-012 reference set.

Below is a template to start your own database file:

In [ ]:
# ── Create an empty database with the correct schema ──────────────────
import pandas as pd
from pathlib import Path

# The 12 column names — must match exactly what fetch_hs_entry.py writes
FIELDS = [
    "Hydrazine synthase name",
    "GenBank ID",
    "Accession Code",
    "UniProt ID",
    "Organism",
    "Year Deposited",
    "Reference(s)",
    "Domain Architecture",
    "Sequence Length",
    "Cupin domain range",
    "MetRS domain range",
    "Sequence",
]

# Create an empty DataFrame with those columns
my_db = pd.DataFrame(columns=FIELDS)

# ── Choose a path for your own database ────────────────────────────────
# Change this path to wherever you want to save your data
MY_DB_PATH = "/scratch/p318738/SSA/AAR-COMP-013/my_HS_database.csv"

# Save — this creates the file with just the header row
my_db.to_csv(MY_DB_PATH, index=False)
print("Empty database created at: {}".format(MY_DB_PATH))

# To add your first entry from the terminal:
print()
print("To add an entry, run in the terminal:")
print("  python /scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step1/fetch_hs_entry.py <ACCESSION> <NAME>")

---

## Summary

In this notebook you:
- Learned the basic Python building blocks: variables, lists, dictionaries, and functions
- Understood CSV files and how to work with them using pandas
- Explored the 24-entry `HS_database.csv` — 10 di-domain, 7 cupin-only, 7 MetRS-only enzymes
- Fetched a live GenBank record with Biopython and read its domain annotations
- Created an empty database template for your own entries

**Next:** Notebook 2 will take those sequences and separate the cupin and MetRS domains into individual FASTA files.

---

## Try it yourself

**Exercise 1:** Load `HS_database.csv` with pandas and print how many Di-domain enzymes there are.

```python
# Hint: use .value_counts() or filter with == "Di-domain" and then len()
db = pd.read_csv("/scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step1/HS_database.csv")
# your code here
```

**Exercise 2:** Fetch accession `BAW27704.1` (Spb40) and print the organism and sequence length.

```python
# Hint: use fetch_record() defined above
# your code here
```

**Exercise 3:** Create a small CSV with three columns (`Name`, `Organism`, `Sequence`) and add 2 made-up entries. Save it to `/tmp/practice.csv` and read it back to verify.

```python
# Hint: pd.DataFrame({'col1': [...], 'col2': [...]}).to_csv(...)
# your code here
```